# Module 3 Lab 3: Textbook Problem 2.2

### Outliers in the Bridges Data

**Name:** Mac Mckay

**Course:** CS 82A

**Repo:** https://github.com/raulmckay-blip/cs82a-portfolio

## Overview

This lab looks through the `bridges` dataset for values that don't seem believable in its one numeric column, `length`. Unlike Lab 2 with the USArrests data, there's no clear rule here. A bridge length in feet doesn't have an obvious impossible range the way a percentage does.

In [ ]:
import pandas as pd

df = pd.read_csv("bridges.csv")
print(df.shape)
df.head()

(108, 13)


,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
0,E1,M,3.0,1818,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
1,E2,A,25.0,1819,HIGHWAY,1037.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
2,E3,A,39.0,1829,AQUEDUCT,NaN,1.0,N,THROUGH,WOOD,NaN,S,WOOD
3,E5,A,29.0,1837,HIGHWAY,1000.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
4,E6,M,23.0,1838,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD


## Summary Statistics

In [ ]:
df["length"].describe()

count      81.000000
mean     1567.469136
std       747.491523
min       804.000000
25%      1000.000000
50%      1300.000000
75%      2000.000000
max      4558.000000
Name: length, dtype: float64

## My Assumption 

Since I don't really know what counts as a normal bridge length in this dataset. It has highway, railroad, and aqueduct bridges built between 1818 and 1986. So I'm using a statistics based rule: a `length` more than **1.5 times the IQR above the 75th percentile** is unusual enough in this sample to flag for a second look. 

In [ ]:
q1, q3 = df["length"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f"Flagging any length above {upper:.1f} ft")
df[df["length"] > upper]

Flagging any length above 3500.0 ft


,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
31,E34,O,41.0,1888,RR,4558.0,2.0,G,THROUGH,STEEL,LONG,F,SIMPLE-T
44,E46,A,37.0,1897,RR,4000.0,2.0,G,DECK,STEEL,LONG,F,SIMPLE-T
104,E91,O,44.0,1975,HIGHWAY,3756.0,6.0,G,THROUGH,STEEL,LONG,F,ARCH


## Result

This rule flags **3 rows Id's**: `E34`, `E46`, and `E91`.

Before deciding what to do with them, I checked if anything else in the data backs up these values. The dataset has a separate column called `span`, with values `SHORT`, `MEDIUM`, or `LONG`. All three flagged bridges are also labeled `span = LONG`, and that label wasn't calculated from `length` at all, it's its own column. Two different columns agreeing is a good sign these are real long bridges and not typos.

In [ ]:
df.loc[df["length"] > upper, ["Id", "river", "erected", "purpose", "length", "span"]]

,Id,river,erected,purpose,length,span
31,E34,O,1888,RR,4558.0,LONG
44,E46,A,1897,RR,4000.0,LONG
104,E91,O,1975,HIGHWAY,3756.0,LONG


## Deliverable Summary

**My assumption:** A `length` value more than 1.5 times the IQR above the 75th percentile, which comes out to above 3,500 ft here, is unusual enough in this sample to take a second look at.

**Flagged rows:** `E34` (4,558 ft), `E46` (4,000 ft), `E91` (3,756 ft).

**What I did:** I kept all three values as they are. Each one is backed up by the dataset's own `span` column, which separately marks all three as `LONG`, a label that wasn't calculated from `length`. Since two different columns agree, I'm treating these as real bridges instead of mistakes. That's the same reasoning I used for South Carolina's assault rate in Lab 2. It's unusual but not impossible, and taking out real extreme values would hide real differences in the data instead of cleaning it up.

## Note: Missing Values

Separate from everything above, `describe()` also shows that only 81 out of 108 rows actually have a `length` value. The other 27 are missing. That's not a bad or unbelievable value, it's just data that isn't there, so I'm not flagging or changing it. Still worth keeping in mind, since it means the length check above only covers about ***75 percent of the dataset.***